# 💊 HỆ THỐNG NHẬN DIỆN THUỐC VÀ ĐỐI SOÁT TƯƠNG TÁC DƯỢC LÝ
### Multiple Pill Recognition & Clinical Drug-Drug Interaction Safety Platform
---
Notebook này tự động thiết lập và chạy toàn bộ hệ thống gồm:
1. **Mô hình AI Computer Vision**: YOLOv11 Segmentation + ResNet-18 Attribute Multi-Head + PaddleOCR GPU
2. **Cơ sở dữ liệu Dược thư Quốc gia (RxNorm CSDL)**: Định danh thuốc, mã khắc 2 mặt, hoạt chất, ma trận DDI (Từ Kaggle `trannhattruong19691/database-mliotlab`)
3. **Thuật toán RAG & Safety Gate**: Tối ưu hóa siêu tham số chuẩn khoa học (Precision 87.5%, Max F1, User Friction 1.98%)
4. **Giao diện Web & Mobile Trực Tuyến**: Streamlit + Cloudflare Public Tunnel (Truy cập trực tiếp không cần mật khẩu)
---
## 🔹 BƯỚC 1: Clone Mã Nguồn (Nhánh tune_rag) & Cài Đặt Thư Viện AI

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# 1. Tự động nhận diện môi trường Google Colab hoặc Kaggle
work_dir = '/content' if os.path.exists('/content') else '/kaggle/working'
repo_dir = os.path.join(work_dir, 'repo')

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

# 2. Clone đúng nhánh tune_rag chứa thuật toán và siêu tham số tối ưu mới nhất
!git clone --depth 1 -b tune_rag https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git {repo_dir}

# 3. Chuyển vào thư mục repo
%cd {repo_dir}

def install_with_retry(args, label, attempts=3):
    """Install a package group and fail clearly after network retries."""
    for attempt in range(1, attempts + 1):
        print(f'[{attempt}/{attempts}] {label}...')
        completed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--timeout', '120', '--retries', '4', *args])
        if completed.returncode == 0:
            return
        if attempt < attempts:
            wait_seconds = 10 * attempt
            print(f'Install failed; retrying in {wait_seconds} seconds.')
            time.sleep(wait_seconds)
    raise RuntimeError(f'{label} failed after {attempts} attempts.')

# 4. Gỡ các package gây xung đột cũ
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu', 'paddleocr', 'paddlex'], check=False)

# 5. Cài đặt các thư viện từ requirements.txt
requirements = Path('requirements.txt').read_text(encoding='utf-8').splitlines()
non_paddle_requirements = [
    line for line in requirements
    if not line.strip().startswith(('paddleocr', 'paddlex'))
]
requirements_without_paddle = Path('/tmp/requirements_without_paddle.txt')
requirements_without_paddle.write_text('\n'.join(non_paddle_requirements) + '\n', encoding='utf-8')
install_with_retry(['-r', str(requirements_without_paddle)], 'Installing project requirements', attempts=2)

# 6. Cài đặt Paddle GPU 3.0.0 & PaddleOCR v3.0.3 chính hãng từ official cu118 index theo chuẩn của đồng đội
install_with_retry([
    'paddlepaddle-gpu==3.0.0',
    '--index-url', 'https://www.paddlepaddle.org.cn/packages/stable/cu118/',
], 'Installing Paddle GPU', attempts=3)
install_with_retry([
    'paddleocr==3.0.3', 'paddlex==3.0.3',
    'numpy==1.26.4', 'opencv-python-headless==4.10.0.84',
], 'Installing PaddleOCR', attempts=2)

smoke_test = subprocess.run([
    sys.executable, '-c',
    "import paddle, paddleocr, paddlex; paddle.set_device('gpu:0'); print('Paddle GPU ready:', paddle.get_device())",
])
if smoke_test.returncode != 0:
    raise RuntimeError('PaddleOCR installation finished but the GPU smoke test failed.')

print("\n✅ BƯỚC 1 HOÀN TẤT: Đã clone mã nguồn tune_rag và cài đặt môi trường GPU thành công!")

---
## 🔹 BƯỚC 2: Tự Động Tải 3 Datasets Từ Kaggle & Nạp Weights AI

In [ ]:
import os, shutil, glob
from pathlib import Path
import kagglehub

repo_root = Path(os.getcwd())
seg_dir = repo_root / 'models/segmentation_yolov11_full_finetune'
attr_dir = repo_root / 'models/attribute_resnet18_last_blocks_finetune'
db_seed_dir = repo_root / 'database_seed'

seg_dir.mkdir(parents=True, exist_ok=True)
attr_dir.mkdir(parents=True, exist_ok=True)
db_seed_dir.mkdir(parents=True, exist_ok=True)

print("🚀 Đang tự động tải 3 Datasets trực tiếp từ link Kaggle...")

# 1. Tải YOLOv11 Segmentation Model
print("1/3. Đang tải mô hình YOLOv11 Segmentation...")
try:
    seg_download_path = kagglehub.dataset_download('nnphuchcmus/pill-segmentation-model')
    for f in glob.glob(f'{seg_download_path}/**/*.pt', recursive=True):
        target_pt = seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt'
        shutil.copy2(f, target_pt)
        size_mb = os.path.getsize(target_pt) / (1024 * 1024)
        print(f"  ✓ Đã nạp YOLOv11-Seg: {target_pt.name} ({size_mb:.1f} MB)")
        break
except Exception as e:
    print(f"  ⚠️ Tải hub lỗi ({e}), đang quét trong input...")
    for f in glob.glob('/kaggle/input/**/*.pt', recursive=True):
        if 'seg' in f.lower() or 'yolo' in f.lower():
            shutil.copy2(f, seg_dir / 'yolov11m_seg_mediseg_full_finetune_v1.pt')
            print(f"  ✓ Đã nạp từ input: {f}")
            break

# 2. Tải ResNet-18 Attribute Recognition Artifacts
print("2/3. Đang tải mô hình ResNet-18 Attribute...")
try:
    attr_download_path = kagglehub.dataset_download('nnphuchcmus/attrubute-artifact')
    copied_attr = 0
    for f in glob.glob(f'{attr_download_path}/**/*', recursive=True):
        if os.path.isfile(f):
            dest = attr_dir / os.path.basename(f)
            shutil.copy2(f, dest)
            copied_attr += 1
    print(f"  ✓ Đã nạp {copied_attr} files cấu hình & weights cho ResNet-18!")
except Exception as e:
    print(f"  ⚠️ Tải hub lỗi ({e})")

# 3. Tải CSDL Dược thư Quốc gia Mới Nhất từ trannhattruong19691/database-mliotlab
print("3/3. Đang tải CSDL Dược thư Quốc gia mới nhất...")
try:
    db_download_path = kagglehub.dataset_download('trannhattruong19691/database-mliotlab')
    db_count = 0
    for f in glob.glob(f'{db_download_path}/**/*', recursive=True):
        if f.endswith('.json'):
            dest = db_seed_dir / os.path.basename(f)
            shutil.copy2(f, dest)
            db_count += 1
        elif f.endswith('.db') or f.endswith('.sqlite'):
            shutil.copy2(f, repo_root / 'medication.db')
            print(f"  ✓ Đã nạp file SQLite database: {os.path.basename(f)}")
    print(f"  ✓ Đã đồng bộ {db_count} file JSON dược thư từ Kaggle Dataset trannhattruong19691/database-mliotlab!")
except Exception as e:
    print(f"  ⚠️ Dùng database_seed mặc định trong repo.")

# 4. Cấu hình file .env cho SQLite
with open(repo_root / '.env', 'w', encoding='utf-8') as f:
    f.write('DATABASE_URL=sqlite:///./medication.db\n')
    f.write('LLM_PROVIDER=fallback\n')

print("\n✅ BƯỚC 2 HOÀN TẤT: Toàn bộ Model AI và CSDL đã sẵn sàng trên đĩa!")

---
## 🔹 BƯỚC 3: Khởi Tạo & Nạp CSDL Dược Thư (SQLite Database)

In [ ]:
import sys, os
from pathlib import Path

repo_root = Path(os.getcwd())
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.environ['PYTHONPATH'] = f"{str(src_path)}:{os.environ.get('PYTHONPATH', '')}"

# Chạy seed database đồng bộ dữ liệu
!python scripts/seed_database.py

from pill_safety.database.session import SessionLocal
from pill_safety.database.models import DrugProduct, DrugInteraction

with SessionLocal() as db:
    total_drugs = db.query(DrugProduct).count()
    total_ddi = db.query(DrugInteraction).count()
    print("=" * 65)
    print(f"📊 CSDL ĐÃ NẠP THÀNH CÔNG: {total_drugs} sản phẩm thuốc | {total_ddi} cặp tương tác DDI")
    print("=" * 65)

print("✅ BƯỚC 3 HOÀN TẤT: CSDL đã sẵn sàng phục vụ truy xuất RAG!")

---
## 🔹 BƯỚC 4: Khởi Chạy Web & Mobile UI (Cloudflare Public Tunnel)

In [ ]:
import subprocess, time, os
from pathlib import Path

# 1. Tắt các tiến trình Streamlit cũ nếu có
!fuser -k 8501/tcp 2>/dev/null || true

# 2. Tải và cài đặt Cloudflared tự động
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

# 3. Khởi chạy Streamlit ở background
cmd_streamlit = "streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false --server.fileWatcherType none"
subprocess.Popen(cmd_streamlit, shell=True)
time.sleep(3)

print("🚀 Streamlit Web App đã khởi chạy thành công trên GPU!")
print("🌐 Đang mở đường link Cloudflare Public Tunnel...")
print("👉 Click trực tiếp vào đường link https://*.trycloudflare.com bên dưới để mở giao diện ứng dụng:\n")

# 4. Mở tunnel Cloudflare trực tiếp
!cloudflared tunnel --url http://localhost:8501